# 05 — EfficientNetB0 (Member 4)

Transfer-learning model for waste image classification (6 classes).

**Owner:** Member 4
**Approach:** ImageNet-pretrained EfficientNetB0 backbone, frozen head training, then optional fine-tuning of the top backbone layers.

This notebook reuses the exact same dataset split, seed, image size, and batch size as the rest of the group (see `01_EDA_Preprocessing.ipynb`), so results are directly comparable with the Custom CNN, MobileNetV2 and ResNet50 notebooks.

> **Preprocessing note (read before running):** `EfficientNetB0` in `tf.keras.applications` has its own `Rescaling` + `Normalization` layers built in and expects raw pixel values in **[0, 255]**. Unlike the shared EDA notebook (which divides by 255 for the other models), this notebook intentionally skips that division and feeds EfficientNetB0 raw resized pixels. This is a documented, architecture-specific deviation — flag it in the report and in the viva.

In [ ]:
from pathlib import Path
import os
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("TensorFlow version:", tf.__version__)

## 1. Configuration

Must match the values used by the rest of the group (see `01_EDA_Preprocessing.ipynb`).

In [ ]:
RANDOM_SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

CLASSES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]
NUM_CLASSES = len(CLASSES)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Random seed:", RANDOM_SEED)

## 2. Paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Garbage_Dataset_Classification"
    / "images"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
TABLES_DIR = RESULTS_DIR / "tables"

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset dir exists:", DATASET_DIR.exists())
print("Processed dir exists:", PROCESSED_DIR.exists())

## 3. Load the shared train / validation / test split

Same CSVs used by every member — do not regenerate or reshuffle these.

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

for split_df in (train_df, validation_df, test_df):
    split_df["image_path"] = split_df["relative_path"].apply(
        lambda x: str(DATASET_DIR / x)
    )

print("Training images:", len(train_df))
print("Validation images:", len(validation_df))
print("Test images:", len(test_df))

In [ ]:
missing_train = sum(not Path(p).exists() for p in train_df["image_path"])
missing_val = sum(not Path(p).exists() for p in validation_df["image_path"])
missing_test = sum(not Path(p).exists() for p in test_df["image_path"])

print("Missing training images:", missing_train)
print("Missing validation images:", missing_val)
print("Missing test images:", missing_test)

assert missing_train == 0 and missing_val == 0 and missing_test == 0, \
    "Some image files referenced in the split CSVs are missing on disk."


## 4. Build the `tf.data` pipeline

**No `/255.0` division here** — EfficientNetB0 expects raw [0, 255] pixel values and rescales internally. This is the one deliberate difference from the shared EDA notebook's `load_and_resize_image` function.

In [ ]:
def create_dataset(dataframe, shuffle=False):
    image_paths = dataframe["image_path"].values
    labels = dataframe["label"].values

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True,
        )

    return dataset


def load_and_resize_image_raw(image_path, label):
    """Load + resize only. Deliberately NOT normalized to [0,1] —
    EfficientNetB0 has its own Rescaling/Normalization layers and
    expects raw [0, 255] pixel values."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)  # stays in [0, 255]
    return image, label

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="train_augmentation")

# Same augmentation settings as the shared EDA notebook, kept identical
# across members for a fair comparison. Applied to TRAINING data only.

In [ ]:
train_dataset = create_dataset(train_df, shuffle=True)
train_dataset = train_dataset.map(load_and_resize_image_raw, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE,
)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

validation_dataset = create_dataset(validation_df, shuffle=False)
validation_dataset = validation_dataset.map(load_and_resize_image_raw, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = create_dataset(test_df, shuffle=False)
test_dataset = test_dataset.map(load_and_resize_image_raw, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Pipelines ready.")

In [ ]:
# Sanity check: confirm pixel range is [0, 255] as EfficientNetB0 expects
images, labels = next(iter(train_dataset))
print("Batch shape:", images.shape)
print("Min pixel value:", tf.reduce_min(images).numpy())
print("Max pixel value:", tf.reduce_max(images).numpy())

plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(CLASSES[labels[i].numpy()])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Build the model

EfficientNetB0 backbone (ImageNet weights, frozen) → Global Average Pooling → Dropout → Dense(6, softmax).

In [ ]:
def build_model(num_classes, image_size, trainable_backbone=False):
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=image_size + (3,),
    )
    base_model.trainable = trainable_backbone

    inputs = keras.Input(shape=image_size + (3,))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="efficientnetb0_waste_classifier")
    return model, base_model


model, base_model = build_model(NUM_CLASSES, IMAGE_SIZE, trainable_backbone=False)
model.summary()

## 6. Phase 1 — train the new classification head (backbone frozen)

In [ ]:
LR_HEAD = 1e-3
EPOCHS_HEAD = 20

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

head_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    keras.callbacks.ModelCheckpoint(
        str(MODELS_DIR / "efficientnetb0_head_best.keras"),
        monitor="val_loss",
        save_best_only=True,
    ),
]

start_time = time.time()
history_head = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_HEAD,
    callbacks=head_callbacks,
)
head_train_time = time.time() - start_time

print(f"Phase 1 training time: {head_train_time:.1f} seconds")

In [ ]:
def plot_history(history, title_suffix, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["accuracy"], label="train")
    axes[0].plot(history.history["val_accuracy"], label="val")
    axes[0].set_title(f"Accuracy {title_suffix}")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["loss"], label="train")
    axes[1].plot(history.history["val_loss"], label="val")
    axes[1].set_title(f"Loss {title_suffix}")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()


plot_history(
    history_head,
    "(EfficientNetB0 — head training)",
    FIGURES_DIR / "efficientnetb0_history_head.png",
)

## 7. Phase 2 — fine-tune the top of the backbone

Unfreeze the top layers of EfficientNetB0 and continue training with a much smaller learning rate, so the pretrained features aren't destroyed. Set `EPOCHS_FINETUNE = 0` to skip this phase if you want a frozen-only result to compare against.

In [ ]:
LR_FINETUNE = 1e-5
EPOCHS_FINETUNE = 10
FINE_TUNE_AT_LAYER = -30  # unfreeze this many layers from the end of the backbone

history_finetune = None
finetune_train_time = 0.0

if EPOCHS_FINETUNE > 0:
    base_model.trainable = True
    for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR_FINETUNE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    finetune_callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            str(MODELS_DIR / "efficientnetb0_finetuned_best.keras"),
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

    start_time = time.time()
    history_finetune = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS_FINETUNE,
        callbacks=finetune_callbacks,
    )
    finetune_train_time = time.time() - start_time

    print(f"Phase 2 (fine-tune) training time: {finetune_train_time:.1f} seconds")

    plot_history(
        history_finetune,
        "(EfficientNetB0 — fine-tuning)",
        FIGURES_DIR / "efficientnetb0_history_finetune.png",
    )
else:
    print("Fine-tuning skipped (EPOCHS_FINETUNE = 0).")

## 8. Final evaluation on the untouched test set

Run this once, after all development/tuning decisions are frozen.

In [ ]:
y_true, y_pred = [], []

for batch_images, batch_labels in test_dataset:
    preds = model.predict(batch_images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(batch_labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

test_accuracy = accuracy_score(y_true, y_pred)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
report_text = classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Macro precision: {macro_precision:.4f}")
print(f"Macro recall: {macro_recall:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print()
print(report_text)

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASSES, yticklabels=CLASSES,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("EfficientNetB0 — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficientnetb0_confusion_matrix.png", dpi=150)
plt.show()

## 9. Save results for the group's comparison table

Saved into `results/metrics`, `results/figures`, `results/tables` — the shared locations used by all four members.

In [ ]:
total_params = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
total_train_time = head_train_time + finetune_train_time

metrics = {
    "model": "EfficientNetB0",
    "member": "Member 4",
    "seed": RANDOM_SEED,
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "epochs_head": len(history_head.history["loss"]),
    "epochs_finetune": len(history_finetune.history["loss"]) if history_finetune else 0,
    "lr_head": LR_HEAD,
    "lr_finetune": LR_FINETUNE if EPOCHS_FINETUNE > 0 else None,
    "fine_tune_at_layer": FINE_TUNE_AT_LAYER if EPOCHS_FINETUNE > 0 else None,
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "total_params": int(total_params),
    "trainable_params_final": trainable_params,
    "training_time_seconds": round(total_train_time, 1),
}

METRICS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

with open(METRICS_DIR / "efficientnetb0_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open(TABLES_DIR / "efficientnetb0_classification_report.txt", "w") as f:
    f.write(report_text)

pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(
    TABLES_DIR / "efficientnetb0_confusion_matrix.csv"
)

model.save(MODELS_DIR / "efficientnetb0_final.keras")

print("Saved:")
print(" -", METRICS_DIR / "efficientnetb0_metrics.json")
print(" -", TABLES_DIR / "efficientnetb0_classification_report.txt")
print(" -", TABLES_DIR / "efficientnetb0_confusion_matrix.csv")
print(" -", MODELS_DIR / "efficientnetb0_final.keras")
print()
print(json.dumps(metrics, indent=2))

## 10. Error analysis

Pull misclassified test images and look for patterns — required by the assignment's Critical Analysis section and useful for comparing failure modes against the other three models.

In [ ]:
misclassified_idx = np.where(y_true != y_pred)[0]
print("Total misclassified:", len(misclassified_idx), "/", len(y_true))

# Confusion pairs, sorted by frequency
from collections import Counter
pairs = Counter(
    (CLASSES[t], CLASSES[p])
    for t, p in zip(y_true[misclassified_idx], y_pred[misclassified_idx])
)
print("\nMost common confusions (true -> predicted):")
for (true_c, pred_c), count in pairs.most_common(10):
    print(f"  {true_c} -> {pred_c}: {count}")

In [ ]:
# Visualize a sample of misclassified images
test_df_reset = test_df.reset_index(drop=True)
sample_idx = np.random.RandomState(RANDOM_SEED).choice(
    misclassified_idx, size=min(9, len(misclassified_idx)), replace=False
)

plt.figure(figsize=(12, 12))
for i, idx in enumerate(sample_idx):
    img_path = test_df_reset.loc[idx, "image_path"]
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE) / 255.0  # just for display

    plt.subplot(3, 3, i + 1)
    plt.imshow(img.numpy())
    plt.title(f"True: {CLASSES[y_true[idx]]}\nPred: {CLASSES[y_pred[idx]]}", fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficientnetb0_misclassified_examples.png", dpi=150)
plt.show()

## 11. Notes for the report / viva

- **Architecture**: EfficientNetB0 (ImageNet-pretrained) → GlobalAveragePooling2D → Dropout(0.3) → Dense(6, softmax).
- **Two-phase training**: frozen-backbone head training, then fine-tuning of the last `abs(FINE_TUNE_AT_LAYER)` backbone layers at a 100x lower learning rate.
- **Preprocessing deviation (documented)**: unlike the other three models, EfficientNetB0 receives raw [0, 255] pixel input rather than the shared `/255.0` normalization, because it has its own internal `Rescaling`/`Normalization` layers. Same split, seed, image size, batch size, and augmentation policy as the rest of the group are otherwise unchanged.
- **What to discuss**: compound scaling as EfficientNet's core idea vs. ResNet's depth-only and MobileNet's width/efficiency focus; whether fine-tuning improved val/test performance over the frozen-only version; parameter count vs. ResNet50 and MobileNetV2; which classes were most confused and why (cross-reference with the other three members' confusion matrices).